# Fine-tune BAAI/bge-m3 cho QA y tế tiếng Việt — LoRA dense → merge full model

Notebook này thực hiện **một pipeline duy nhất**:

```text
BAAI/bge-m3
→ SentenceTransformers + PEFT LoRA
→ MultipleNegativesRankingLoss
→ merge LoRA vào base model bằng PEFT merge_and_unload()
→ save full merged SentenceTransformers artifact
→ reload local gate
→ push full folder lên Hugging Face Hub
→ reload Hub gate
→ final benchmark
```

Mục tiêu output cuối:

```python
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("nqd-301125/bge-m3-medical-vi-dense")
emb = model.encode(["Đau đầu kéo dài có nguy hiểm không?"], normalize_embeddings=True)
```

Các điểm khóa cứng:

- Không dùng `unsloth`.
- Không dùng `save_pretrained_merged()` / `push_to_hub_merged()`.
- Không push adapter-only repo.
- Chỉ push sau khi **local reload gate** pass.
- Repo Hub cũ sẽ được xóa file cũ bằng `delete_patterns=["*"]` trước khi upload folder mới, để tránh sót artifact cũ.

## Điều kiện để Run all một lần là xong

Trước khi bấm **Run all**, cần có Hugging Face write token ở một trong các nơi sau:

1. Environment variable `HF_TOKEN`; hoặc
2. Colab Secret tên `HF_TOKEN`.

Notebook sẽ kiểm tra quyền ghi repo Hub ngay đầu pipeline. Nếu không có token hoặc token không có quyền push vào `REPO_ID`, notebook sẽ dừng sớm trước khi train để tránh mất thời gian.


## 00. Cấu hình đã chốt

Thông số chốt theo điều kiện GPU bạn đã xác nhận:

| Thành phần | Giá trị |
|---|---|
| Base model | `BAAI/bge-m3` |
| Dataset | `nqd-301125/medical_data` |
| Output repo | `nqd-301125/bge-m3-medical-vi-dense` |
| Output cuối | Full merged SentenceTransformers model |
| `max_seq_length` | `6000` |
| Batch thật | `256` |
| Gradient accumulation | `1` |
| Epochs | `2` |
| LR | `1e-4` |
| Warmup ratio | `0.10` |
| Weight decay | `0.01` |
| Loss | `MultipleNegativesRankingLoss(scale=20.0)` |
| LoRA target modules | `["query", "key", "value", "dense"]` |
| LoRA r / alpha / dropout | `16 / 32 / 0.05` |
| Split | `90/5/5` |
| Eval corpus | toàn bộ unique positives |
| Best metric | dev `cosine_ndcg@10` |

Ghi chú: pipeline này fine-tune **dense embedding behavior**. Không claim rằng sparse/multi-vector path của BGE-M3 đã được fine-tune.

In [ ]:
# 01_install_dependencies
# Chạy cell này trong runtime sạch của Colab.
# Không cài Unsloth. Nếu môi trường có Unsloth từ trước thì gỡ ra để tránh nhầm pipeline.

import sys
import subprocess
import os

def run_cmd(cmd, check=True):
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, check=check)
    return result

# Gỡ các package không dùng trong pipeline mới.
run_cmd([sys.executable, "-m", "pip", "uninstall", "-y", "unsloth", "unsloth_zoo", "trl"], check=False)

# Chỉ pin sentence-transformers theo bản docs/release đã kiểm tra.
# Các dependency còn lại để pip resolver chọn bản tương thích với sentence-transformers[train].
# Điều này giảm rủi ro conflict khi Transformers/Hub thay đổi dependency.
run_cmd([
    sys.executable, "-m", "pip", "install", "-U", "--quiet",
    "sentence-transformers[train]==5.5.1",
    "peft",
    "datasets",
    "huggingface_hub",
    "accelerate",
    "safetensors",
])

# Gate môi trường: nếu có conflict thì dừng trước khi train.
run_cmd([sys.executable, "-m", "pip", "check"], check=True)

print("Install completed. Nếu Colab yêu cầu restart runtime sau pip install, hãy restart rồi chạy lại từ cell import/version bên dưới.")

In [ ]:
# 02_imports_versions_and_global_config

import os
import re
import gc
import json
import math
import shutil
import random
import inspect
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch

import transformers
import datasets
import peft
import sentence_transformers
import huggingface_hub

from datasets import load_dataset, Dataset, concatenate_datasets
from peft import LoraConfig, TaskType
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, losses
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from sentence_transformers.util import cos_sim
from sentence_transformers import SentenceTransformerTrainingArguments
from sentence_transformers.training_args import BatchSamplers
from huggingface_hub import HfApi, login, notebook_login, whoami

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("peft:", peft.__version__)
print("sentence_transformers:", sentence_transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("cuda device:", torch.cuda.get_device_name(0))
    print("bf16 supported:", torch.cuda.is_bf16_supported())

# Cấu hình chốt
BASE_MODEL = "BAAI/bge-m3"
DATASET_ID = "nqd-301125/medical_data"
REPO_ID = "nqd-301125/bge-m3-medical-vi-dense"

SEED = 3407

MAX_SEQ_LENGTH = 6000
TRAIN_BATCH_SIZE = 256
EVAL_BATCH_SIZE = 128
GRADIENT_ACCUMULATION_STEPS = 1

NUM_EPOCHS = 2
LEARNING_RATE = 1e-4
WARMUP_RATIO = 0.10
WEIGHT_DECAY = 0.01

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["query", "key", "value", "dense"]

OUTPUT_ROOT = Path("/content/bge_m3_medical_vi_dense_run")
TRAIN_OUTPUT_DIR = OUTPUT_ROOT / "checkpoints"
LOCAL_MERGED_DIR = OUTPUT_ROOT / "bge-m3-medical-vi-dense-full-merged"
METRICS_DIR = OUTPUT_ROOT / "metrics"
HUB_RELOAD_CACHE = OUTPUT_ROOT / "hub_reload_cache"

PUSH_TO_HUB = True
DELETE_REMOTE_FILES_BEFORE_UPLOAD = True
ALLOW_INTERACTIVE_HF_LOGIN = False  # Để Run all không bị treo ở UI login; hãy dùng HF_TOKEN/Colab Secret.

# Tăng tốc upload/download Hub nếu huggingface_hub cài hf_xet.
os.environ.setdefault("HF_XET_HIGH_PERFORMANCE", "1")

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# dtype chốt: bf16 nếu GPU hỗ trợ, ngược lại fp16 trên CUDA, fp32 trên CPU.
if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    TORCH_DTYPE = torch.bfloat16
    USE_BF16 = True
    USE_FP16 = False
elif torch.cuda.is_available():
    TORCH_DTYPE = torch.float16
    USE_BF16 = False
    USE_FP16 = True
else:
    TORCH_DTYPE = torch.float32
    USE_BF16 = False
    USE_FP16 = False

print("TORCH_DTYPE:", TORCH_DTYPE)
print("USE_BF16:", USE_BF16, "USE_FP16:", USE_FP16)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
TRAIN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

# Hard check: pipeline này không import Unsloth.
assert "unsloth" not in globals(), "Do not use Unsloth in this notebook."

In [ ]:
# 03_huggingface_login_and_repo_write_preflight
# Để "Run all" thật sự chạy một mạch, hãy đặt Hugging Face write token vào:
#   - environment variable HF_TOKEN; hoặc
#   - Colab Secret tên HF_TOKEN.
# Nếu không có token, notebook sẽ dừng sớm trước khi train.

def get_hf_token_for_run_all():
    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACEHUB_API_TOKEN")
    if token:
        return token, "environment"

    # Colab Secrets support.
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            return token, "colab_secret:HF_TOKEN"
    except Exception:
        pass

    return None, None

HF_TOKEN, HF_TOKEN_SOURCE = get_hf_token_for_run_all()

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
elif ALLOW_INTERACTIVE_HF_LOGIN:
    print("No HF token found. Opening notebook_login(). This is not one-click Run all mode.")
    notebook_login()
else:
    raise RuntimeError(
        "No Hugging Face token found. For one-click Run all, set a write token as HF_TOKEN "
        "in environment variables or Colab Secrets before running the notebook."
    )

hf_user = whoami()
print("HF logged in as:", hf_user.get("name") or hf_user)
print("HF token source:", HF_TOKEN_SOURCE or "interactive")

# Preflight quyền ghi Hub trước khi train để không mất thời gian nếu token/repo sai.
if PUSH_TO_HUB:
    api = HfApi()
    api.create_repo(repo_id=REPO_ID, repo_type="model", exist_ok=True)
    print("Hub repo write preflight passed:", REPO_ID)


In [ ]:
# 04_load_dataset_and_resolve_columns

raw = load_dataset(DATASET_ID)
print(raw)

if isinstance(raw, datasets.DatasetDict):
    split_name = "train" if "train" in raw else list(raw.keys())[0]
    ds_raw = raw[split_name]
else:
    split_name = "dataset"
    ds_raw = raw

print("Using split:", split_name)
print("Rows:", len(ds_raw))
print("Columns:", ds_raw.column_names)
print("First example:", ds_raw[0])

def resolve_column(columns, candidates):
    normalized = {c.lower().strip(): c for c in columns}
    for cand in candidates:
        if cand.lower() in normalized:
            return normalized[cand.lower()]
    return None

ANCHOR_COLUMN = resolve_column(
    ds_raw.column_names,
    ["anchor", "question", "query", "instruction", "input", "prompt", "cau_hoi", "hoi"]
)
POSITIVE_COLUMN = resolve_column(
    ds_raw.column_names,
    ["positive", "answer", "response", "output", "context", "document", "passage", "tra_loi", "dap_an"]
)

if ANCHOR_COLUMN is None or POSITIVE_COLUMN is None:
    raise ValueError(
        f"Cannot resolve anchor/positive columns. Columns={ds_raw.column_names}. "
        "Set ANCHOR_COLUMN and POSITIVE_COLUMN manually in this cell."
    )

print("ANCHOR_COLUMN:", ANCHOR_COLUMN)
print("POSITIVE_COLUMN:", POSITIVE_COLUMN)

ds_pairs = ds_raw.select_columns([ANCHOR_COLUMN, POSITIVE_COLUMN])
rename_map = {}
if ANCHOR_COLUMN != "anchor":
    rename_map[ANCHOR_COLUMN] = "anchor"
if POSITIVE_COLUMN != "positive":
    rename_map[POSITIVE_COLUMN] = "positive"
if rename_map:
    ds_pairs = ds_pairs.rename_columns(rename_map)

print(ds_pairs)
print(ds_pairs[0])

In [ ]:
# 05_clean_and_deduplicate_dataset

MIN_ANCHOR_CHARS = 10
MIN_POSITIVE_CHARS = 30

def normalize_text(x):
    if x is None:
        return ""
    x = str(x)
    x = x.replace("\u00a0", " ")
    x = re.sub(r"\s+", " ", x)
    return x.strip()

def normalize_for_dedup(x):
    x = normalize_text(x).lower()
    x = re.sub(r"\s+", " ", x)
    return x

def clean_example(ex):
    return {
        "anchor": normalize_text(ex["anchor"]),
        "positive": normalize_text(ex["positive"]),
    }

before = len(ds_pairs)
ds_clean = ds_pairs.map(clean_example)

ds_clean = ds_clean.filter(
    lambda x: len(x["anchor"]) >= MIN_ANCHOR_CHARS and len(x["positive"]) >= MIN_POSITIVE_CHARS
)

after_len_filter = len(ds_clean)

df = ds_clean.to_pandas()
df["anchor_norm"] = df["anchor"].map(normalize_for_dedup)
df["positive_norm"] = df["positive"].map(normalize_for_dedup)
df["pair_norm"] = df["anchor_norm"] + " ||| " + df["positive_norm"]

dup_pairs = int(df["pair_norm"].duplicated().sum())
dup_anchors = int(df["anchor_norm"].duplicated().sum())
dup_positives = int(df["positive_norm"].duplicated().sum())

print("Before:", before)
print("After min length filter:", after_len_filter)
print("Duplicate exact pairs:", dup_pairs)
print("Duplicate exact anchors:", dup_anchors)
print("Duplicate exact positives:", dup_positives)

# Chính sách chốt cho một lần chạy: ưu tiên dataset sạch để giảm false negatives với MNRL.
df = df.drop_duplicates("pair_norm", keep="first")
df = df.drop_duplicates("anchor_norm", keep="first")
df = df.drop_duplicates("positive_norm", keep="first")

df = df[["anchor", "positive"]].reset_index(drop=True)
ds_clean = Dataset.from_pandas(df, preserve_index=False)

print("After dedup:", len(ds_clean))
assert len(ds_clean) >= 1000, "Dataset sau clean quá nhỏ. Kiểm tra lại columns hoặc cleaning rules."

print(ds_clean[0])

In [ ]:
# 06_split_and_leakage_check

tmp = ds_clean.train_test_split(test_size=0.10, seed=SEED, shuffle=True)
val_test = tmp["test"].train_test_split(test_size=0.50, seed=SEED, shuffle=True)

train_ds = tmp["train"]
dev_ds = val_test["train"]
test_ds = val_test["test"]

print("train:", len(train_ds))
print("dev:", len(dev_ds))
print("test:", len(test_ds))

def norm_set(dataset, col):
    return {normalize_for_dedup(x[col]) for x in dataset}

def pair_set(dataset):
    return {
        normalize_for_dedup(x["anchor"]) + " ||| " + normalize_for_dedup(x["positive"])
        for x in dataset
    }

sets = {
    "train_anchor": norm_set(train_ds, "anchor"),
    "dev_anchor": norm_set(dev_ds, "anchor"),
    "test_anchor": norm_set(test_ds, "anchor"),
    "train_positive": norm_set(train_ds, "positive"),
    "dev_positive": norm_set(dev_ds, "positive"),
    "test_positive": norm_set(test_ds, "positive"),
    "train_pair": pair_set(train_ds),
    "dev_pair": pair_set(dev_ds),
    "test_pair": pair_set(test_ds),
}

checks = {
    "anchor train/dev": len(sets["train_anchor"] & sets["dev_anchor"]),
    "anchor train/test": len(sets["train_anchor"] & sets["test_anchor"]),
    "anchor dev/test": len(sets["dev_anchor"] & sets["test_anchor"]),
    "positive train/dev": len(sets["train_positive"] & sets["dev_positive"]),
    "positive train/test": len(sets["train_positive"] & sets["test_positive"]),
    "positive dev/test": len(sets["dev_positive"] & sets["test_positive"]),
    "pair train/dev": len(sets["train_pair"] & sets["dev_pair"]),
    "pair train/test": len(sets["train_pair"] & sets["test_pair"]),
    "pair dev/test": len(sets["dev_pair"] & sets["test_pair"]),
}

print(json.dumps(checks, indent=2, ensure_ascii=False))

assert checks["pair train/dev"] == 0
assert checks["pair train/test"] == 0
assert checks["pair dev/test"] == 0
assert checks["anchor train/dev"] == 0
assert checks["anchor train/test"] == 0
assert checks["anchor dev/test"] == 0
assert checks["positive train/dev"] == 0
assert checks["positive train/test"] == 0
assert checks["positive dev/test"] == 0

print("Leakage exact check passed.")

In [ ]:
# 07_helper_functions_for_model_loading_and_eval

def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def load_sentence_model(model_name_or_path, max_seq_length=MAX_SEQ_LENGTH):
    """Load SentenceTransformer with dtype fallback."""
    try:
        model = SentenceTransformer(
            model_name_or_path,
            model_kwargs={"torch_dtype": TORCH_DTYPE},
            trust_remote_code=False,
        )
    except TypeError:
        # Fallback for APIs where model_kwargs/trust_remote_code signature differs.
        model = SentenceTransformer(model_name_or_path)
    model.max_seq_length = max_seq_length
    return model

def build_ir_evaluator(eval_ds, all_corpus_ds, name):
    corpus_texts = list(dict.fromkeys(all_corpus_ds["positive"]))
    corpus = {f"doc-{i}": text for i, text in enumerate(corpus_texts)}
    text_to_cid = {text: cid for cid, text in corpus.items()}

    queries = {}
    relevant_docs = {}
    skipped = 0

    for i, ex in enumerate(eval_ds):
        qid = f"q-{i}"
        cid = text_to_cid.get(ex["positive"])
        if cid is None:
            skipped += 1
            continue
        queries[qid] = ex["anchor"]
        relevant_docs[qid] = {cid}

    assert len(queries) > 0, f"No queries built for evaluator {name}"
    assert skipped == 0, f"Skipped {skipped} examples because positive was not in corpus"

    evaluator = InformationRetrievalEvaluator(
        queries=queries,
        corpus=corpus,
        relevant_docs=relevant_docs,
        name=name,
        accuracy_at_k=[1, 3, 5, 10],
        precision_recall_at_k=[1, 3, 5, 10],
        mrr_at_k=[10],
        ndcg_at_k=[10],
        map_at_k=[100],
        score_functions={"cosine": cos_sim},
        main_score_function="cosine",
        show_progress_bar=True,
        batch_size=EVAL_BATCH_SIZE,
    )
    return evaluator, {"num_queries": len(queries), "num_corpus": len(corpus)}

all_corpus_ds = concatenate_datasets([train_ds, dev_ds, test_ds])

dev_evaluator, dev_eval_info = build_ir_evaluator(dev_ds, all_corpus_ds, "medical_dev")
test_evaluator, test_eval_info = build_ir_evaluator(test_ds, all_corpus_ds, "medical_test")

print("dev_eval_info:", dev_eval_info)
print("test_eval_info:", test_eval_info)

def find_ndcg10_key(metrics):
    keys = list(metrics.keys())
    candidates = [k for k in keys if k.endswith("cosine_ndcg@10")]
    if not candidates:
        candidates = [k for k in keys if "ndcg@10" in k and "cosine" in k]
    if not candidates:
        raise KeyError(f"Cannot find cosine_ndcg@10 in metric keys: {keys}")
    return candidates[0]

def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def smoke_test_model(model, expected_dim=1024, name="model"):
    texts = [
        "Đau đầu kéo dài có nguy hiểm không?",
        "Trẻ bị sốt cao nên xử lý như thế nào?",
    ]
    emb = model.encode(
        texts,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
        batch_size=2,
    )
    print(name, "embedding shape:", emb.shape)
    print(name, "finite:", np.isfinite(emb).all())
    print(name, "norm:", np.linalg.norm(emb, axis=1))

    assert emb.shape == (2, expected_dim), f"{name}: expected shape (2,{expected_dim}), got {emb.shape}"
    assert np.isfinite(emb).all(), f"{name}: embedding contains NaN/Inf"
    return emb

In [ ]:
# 08_baseline_eval_base_bge_m3
# Đánh giá base model trước train để có mốc A/B.
# Nếu muốn tiết kiệm thời gian tuyệt đối, có thể đặt RUN_BASELINE_EVAL=False.
# Tuy nhiên khuyến nghị giữ True để biết finetune có cải thiện thật không.

RUN_BASELINE_EVAL = True

if RUN_BASELINE_EVAL:
    cleanup_cuda()
    base_model = load_sentence_model(BASE_MODEL, max_seq_length=MAX_SEQ_LENGTH)
    print("Base model max_seq_length:", base_model.max_seq_length)

    base_smoke_emb = smoke_test_model(base_model, expected_dim=1024, name="base_model")

    base_dev_metrics = dev_evaluator(base_model)
    base_test_metrics = test_evaluator(base_model)

    print("BASE DEV METRICS")
    print(json.dumps(base_dev_metrics, indent=2, ensure_ascii=False))
    print("BASE TEST METRICS")
    print(json.dumps(base_test_metrics, indent=2, ensure_ascii=False))

    save_json(base_dev_metrics, METRICS_DIR / "base_dev_metrics.json")
    save_json(base_test_metrics, METRICS_DIR / "base_test_metrics.json")

    NDCG10_KEY = find_ndcg10_key(base_dev_metrics)
    METRIC_FOR_BEST_MODEL = f"eval_{NDCG10_KEY}" if not NDCG10_KEY.startswith("eval_") else NDCG10_KEY
    print("NDCG10_KEY:", NDCG10_KEY)
    print("METRIC_FOR_BEST_MODEL:", METRIC_FOR_BEST_MODEL)

    del base_model
    cleanup_cuda()
else:
    # Fallback nếu bỏ baseline. Tên metric này khớp evaluator name và cosine score function.
    NDCG10_KEY = "medical_dev_cosine_ndcg@10"
    METRIC_FOR_BEST_MODEL = "eval_medical_dev_cosine_ndcg@10"
    print("Baseline skipped. Using metric key:", METRIC_FOR_BEST_MODEL)

In [ ]:
# 09_load_train_model_add_lora_and_verify

cleanup_cuda()

model = load_sentence_model(BASE_MODEL, max_seq_length=MAX_SEQ_LENGTH)
print("Train model loaded.")
print("max_seq_length:", model.max_seq_length)

# Kiểm tra target module names trước khi add LoRA.
linear_names = []
for name, module in model[0].auto_model.named_modules():
    if isinstance(module, torch.nn.Linear):
        linear_names.append(name)

matched = [
    name for name in linear_names
    if name.split(".")[-1] in set(LORA_TARGET_MODULES)
]

print("Number of Linear modules:", len(linear_names))
print("Number of matched LoRA target modules:", len(matched))
print("Sample matched modules:")
for n in matched[:30]:
    print("  ", n)

missing_target_kinds = []
for target in LORA_TARGET_MODULES:
    if not any(name.split(".")[-1] == target for name in linear_names):
        missing_target_kinds.append(target)

assert not missing_target_kinds, f"These LoRA target module leaf names were not found: {missing_target_kinds}"
assert len(matched) > 0, "No LoRA target modules matched."

peft_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    inference_mode=False,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules=LORA_TARGET_MODULES,
)

model.add_adapter(peft_config)
model.enable_adapters()

# Verify PEFT model.
auto_model = model[0].auto_model
assert hasattr(auto_model, "merge_and_unload"), (
    "After add_adapter, auto_model should have merge_and_unload(). "
    "If this fails, PEFT adapter was not attached correctly."
)

print("PEFT adapter added.")
if hasattr(auto_model, "print_trainable_parameters"):
    auto_model.print_trainable_parameters()

# Gradient checkpointing chốt.
if hasattr(auto_model, "gradient_checkpointing_enable"):
    auto_model.gradient_checkpointing_enable()
    print("Gradient checkpointing enabled.")

if hasattr(auto_model, "config") and hasattr(auto_model.config, "use_cache"):
    auto_model.config.use_cache = False
    print("Set use_cache=False.")

# Smoke test trước train.
_ = smoke_test_model(model, expected_dim=1024, name="train_model_with_lora")

In [ ]:
# 10_training_arguments_and_trainer

train_loss = losses.MultipleNegativesRankingLoss(
    model=model,
    scale=20.0,
)

# Với batch 256 và dataset ~10k, mỗi epoch khoảng 35-40 steps.
# eval/save mỗi 10 steps để load_best_model_at_end chọn checkpoint tốt theo dev NDCG@10.
num_train_steps_per_epoch = math.ceil(len(train_ds) / TRAIN_BATCH_SIZE)
approx_total_steps = num_train_steps_per_epoch * NUM_EPOCHS

print("Approx steps/epoch:", num_train_steps_per_epoch)
print("Approx total steps:", approx_total_steps)

EVAL_STEPS = 10
SAVE_STEPS = 10
LOGGING_STEPS = 2

# Xử lý khác biệt tên tham số eval_strategy/evaluation_strategy giữa các version Transformers.
training_args_kwargs = dict(
    output_dir=str(TRAIN_OUTPUT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    logging_steps=LOGGING_STEPS,
    load_best_model_at_end=True,
    metric_for_best_model=METRIC_FOR_BEST_MODEL,
    greater_is_better=True,
    bf16=USE_BF16,
    fp16=USE_FP16,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    report_to=[],
    seed=SEED,
    dataloader_drop_last=False,
)

sig = inspect.signature(SentenceTransformerTrainingArguments)
if "eval_strategy" in sig.parameters:
    training_args_kwargs["eval_strategy"] = "steps"
elif "evaluation_strategy" in sig.parameters:
    training_args_kwargs["evaluation_strategy"] = "steps"
else:
    raise RuntimeError("Cannot find eval_strategy/evaluation_strategy in SentenceTransformerTrainingArguments signature.")

training_args_kwargs["eval_steps"] = EVAL_STEPS

args = SentenceTransformerTrainingArguments(**training_args_kwargs)

print(args)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    loss=train_loss,
    evaluator=dev_evaluator,
)

In [ ]:
# 11_train
# Sau cell này, load_best_model_at_end=True phải đưa model về checkpoint tốt nhất theo dev cosine_ndcg@10.

train_result = trainer.train()
print(train_result)

# Lưu trainer state/logs để audit.
trainer_state_path = METRICS_DIR / "trainer_state.json"
try:
    trainer.state.save_to_json(str(trainer_state_path))
    print("Saved trainer state:", trainer_state_path)
except Exception as e:
    print("Could not save trainer state:", repr(e))

# Evaluate lại sau train bằng model đang ở best checkpoint.
post_train_dev_metrics = dev_evaluator(model)
post_train_test_metrics = test_evaluator(model)

print("POST TRAIN DEV METRICS")
print(json.dumps(post_train_dev_metrics, indent=2, ensure_ascii=False))
print("POST TRAIN TEST METRICS")
print(json.dumps(post_train_test_metrics, indent=2, ensure_ascii=False))

save_json(post_train_dev_metrics, METRICS_DIR / "post_train_dev_metrics_before_merge.json")
save_json(post_train_test_metrics, METRICS_DIR / "post_train_test_metrics_before_merge.json")

In [ ]:
# 12_merge_lora_into_base_model
# Đây là bước quan trọng nhất để output cuối KHÔNG phải adapter-only.

model.eval()
cleanup_cuda()

auto_model = model[0].auto_model

print("auto_model type before merge:", type(auto_model))
assert hasattr(auto_model, "merge_and_unload"), (
    "Current auto_model has no merge_and_unload(). "
    "Do not save/push because LoRA may not be attached as expected."
)

# Merge LoRA weights vào base model.
merged_auto_model = auto_model.merge_and_unload()

# Gán lại vì merge_and_unload không nên được giả định là in-place.
model[0].auto_model = merged_auto_model

print("auto_model type after merge:", type(model[0].auto_model))

# Kiểm tra không còn LoRA module trong merged model.
lora_modules = [
    name for name, _ in model[0].auto_model.named_modules()
    if "lora" in name.lower()
]
print("Remaining lora modules:", len(lora_modules))
if lora_modules[:20]:
    print(lora_modules[:20])

assert len(lora_modules) == 0, "LoRA modules still exist after merge. Do not save/push."

# Smoke test sau merge trong RAM.
merged_ram_emb = smoke_test_model(model, expected_dim=1024, name="merged_model_in_ram")

In [ ]:
# 13_save_full_merged_sentence_transformers_artifact

if LOCAL_MERGED_DIR.exists():
    shutil.rmtree(LOCAL_MERGED_DIR)
LOCAL_MERGED_DIR.mkdir(parents=True, exist_ok=True)

# Save bằng SentenceTransformers, không dùng hàm merged của Unsloth.
try:
    model.save_pretrained(str(LOCAL_MERGED_DIR), safe_serialization=True)
except TypeError:
    model.save_pretrained(str(LOCAL_MERGED_DIR))

print("Saved merged model to:", LOCAL_MERGED_DIR)

# Validate file structure.
all_files = [p.relative_to(LOCAL_MERGED_DIR).as_posix() for p in LOCAL_MERGED_DIR.rglob("*") if p.is_file()]
print("Saved files:")
for f in all_files[:200]:
    print(" ", f)
if len(all_files) > 200:
    print(" ...", len(all_files) - 200, "more files")

assert (LOCAL_MERGED_DIR / "modules.json").exists(), "Missing modules.json; not a valid SentenceTransformers artifact."
assert (LOCAL_MERGED_DIR / "config_sentence_transformers.json").exists(), "Missing config_sentence_transformers.json."

adapter_files = [f for f in all_files if "adapter_model" in f or "adapter_config" in f]
assert len(adapter_files) == 0, (
    "Adapter files found in merged output. This is not a clean full merged model: "
    + str(adapter_files)
)

weight_files = [
    f for f in all_files
    if re.search(r"(model|pytorch_model).*\.(safetensors|bin)$", f)
]
print("Weight files:", weight_files)
assert len(weight_files) > 0, "No full model weight files found. Do not push."

with open(LOCAL_MERGED_DIR / "modules.json", "r", encoding="utf-8") as f:
    modules_json_text = f.read()

assert "unsloth" not in modules_json_text.lower(), "modules.json contains Unsloth reference."

print("Full merged artifact file gate passed.")

In [ ]:
# 14_local_reload_gate_and_eval
# Nếu cell này fail thì không được push Hub.

cleanup_cuda()

local_reloaded = load_sentence_model(str(LOCAL_MERGED_DIR), max_seq_length=MAX_SEQ_LENGTH)
local_reloaded_emb = smoke_test_model(local_reloaded, expected_dim=1024, name="local_reloaded_merged_model")

# So sánh embedding model merged trong RAM và model reload từ disk.
max_abs_diff = float(np.max(np.abs(merged_ram_emb - local_reloaded_emb)))
print("Max abs diff between RAM merged and local reloaded smoke embeddings:", max_abs_diff)

# Với bf16/fp16 có thể có sai khác nhỏ, nhưng không được lệch lớn.
assert max_abs_diff < 5e-2, f"Reloaded model differs too much from RAM merged model: {max_abs_diff}"

local_dev_metrics = dev_evaluator(local_reloaded)
local_test_metrics = test_evaluator(local_reloaded)

print("LOCAL MERGED DEV METRICS")
print(json.dumps(local_dev_metrics, indent=2, ensure_ascii=False))
print("LOCAL MERGED TEST METRICS")
print(json.dumps(local_test_metrics, indent=2, ensure_ascii=False))

save_json(local_dev_metrics, METRICS_DIR / "local_merged_dev_metrics.json")
save_json(local_test_metrics, METRICS_DIR / "local_merged_test_metrics.json")

print("Local reload gate passed.")

In [ ]:
# 15_write_model_card_readme

def metric_get(metrics, suffix):
    for k, v in metrics.items():
        if k.endswith(suffix):
            return v
    return None

base_dev_summary = {}
base_test_summary = {}
if RUN_BASELINE_EVAL:
    base_dev_summary = {
        "NDCG@10": metric_get(base_dev_metrics, "cosine_ndcg@10"),
        "MRR@10": metric_get(base_dev_metrics, "cosine_mrr@10"),
        "Recall@10": metric_get(base_dev_metrics, "cosine_recall@10"),
        "Accuracy@1": metric_get(base_dev_metrics, "cosine_accuracy@1"),
        "MAP@100": metric_get(base_dev_metrics, "cosine_map@100"),
    }
    base_test_summary = {
        "NDCG@10": metric_get(base_test_metrics, "cosine_ndcg@10"),
        "MRR@10": metric_get(base_test_metrics, "cosine_mrr@10"),
        "Recall@10": metric_get(base_test_metrics, "cosine_recall@10"),
        "Accuracy@1": metric_get(base_test_metrics, "cosine_accuracy@1"),
        "MAP@100": metric_get(base_test_metrics, "cosine_map@100"),
    }

local_dev_summary = {
    "NDCG@10": metric_get(local_dev_metrics, "cosine_ndcg@10"),
    "MRR@10": metric_get(local_dev_metrics, "cosine_mrr@10"),
    "Recall@10": metric_get(local_dev_metrics, "cosine_recall@10"),
    "Accuracy@1": metric_get(local_dev_metrics, "cosine_accuracy@1"),
    "MAP@100": metric_get(local_dev_metrics, "cosine_map@100"),
}
local_test_summary = {
    "NDCG@10": metric_get(local_test_metrics, "cosine_ndcg@10"),
    "MRR@10": metric_get(local_test_metrics, "cosine_mrr@10"),
    "Recall@10": metric_get(local_test_metrics, "cosine_recall@10"),
    "Accuracy@1": metric_get(local_test_metrics, "cosine_accuracy@1"),
    "MAP@100": metric_get(local_test_metrics, "cosine_map@100"),
}

model_card = f"""---
language:
- vi
license: mit
base_model: {BASE_MODEL}
library_name: sentence-transformers
pipeline_tag: sentence-similarity
tags:
- sentence-transformers
- feature-extraction
- embeddings
- vietnamese
- medical
- bge-m3
- dense-retrieval
---

# bge-m3-medical-vi-dense

Full merged SentenceTransformers model fine-tuned from `{BASE_MODEL}` for Vietnamese medical QA dense retrieval.

## Intended use

This model is intended for dense embedding retrieval over Vietnamese medical QA/content collections.

Example:

```python
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("{REPO_ID}")
emb = model.encode(["Đau đầu kéo dài có nguy hiểm không?"], normalize_embeddings=True)
print(emb.shape)
```

## Training setup

- Base model: `{BASE_MODEL}`
- Dataset: `{DATASET_ID}`
- Training library: `sentence-transformers` + `peft`
- Training method: LoRA, then merged into base model via PEFT `merge_and_unload()`
- Output artifact: full merged SentenceTransformers model
- `max_seq_length`: `{MAX_SEQ_LENGTH}`
- Embedding dimension expected: `1024`
- Loss: `MultipleNegativesRankingLoss(scale=20.0)`
- Epochs: `{NUM_EPOCHS}`
- Batch size: `{TRAIN_BATCH_SIZE}`
- Gradient accumulation: `{GRADIENT_ACCUMULATION_STEPS}`
- LR: `{LEARNING_RATE}`
- Warmup ratio: `{WARMUP_RATIO}`
- Weight decay: `{WEIGHT_DECAY}`
- LoRA r: `{LORA_R}`
- LoRA alpha: `{LORA_ALPHA}`
- LoRA dropout: `{LORA_DROPOUT}`
- LoRA target modules: `{LORA_TARGET_MODULES}`

## Scope

This is a dense retrieval fine-tune. It should not be described as explicitly fine-tuning the sparse or multi-vector functions of BGE-M3.

## Evaluation setup

- Split: train/dev/test = 90/5/5 after exact deduplication
- Evaluator: `InformationRetrievalEvaluator`
- Queries: anchors from dev/test
- Corpus: all unique positives from train + dev + test
- Main metric: cosine NDCG@10

## Local merged metrics

Dev:

```json
{json.dumps(local_dev_summary, ensure_ascii=False, indent=2)}
```

Test:

```json
{json.dumps(local_test_summary, ensure_ascii=False, indent=2)}
```

## Base model metrics

Dev:

```json
{json.dumps(base_dev_summary, ensure_ascii=False, indent=2)}
```

Test:

```json
{json.dumps(base_test_summary, ensure_ascii=False, indent=2)}
```

## Limitations

- This model is not a medical diagnosis system.
- Retrieval results should be reviewed before use in clinical or safety-critical contexts.
- Training data distribution may not represent all medical specialties or real-world user queries.
"""

readme_path = LOCAL_MERGED_DIR / "README.md"
readme_path.write_text(model_card, encoding="utf-8")
print("Wrote model card:", readme_path)

# Copy metrics into artifact folder for reproducibility.
artifact_metrics_dir = LOCAL_MERGED_DIR / "training_metrics"
artifact_metrics_dir.mkdir(exist_ok=True)
for p in METRICS_DIR.glob("*.json"):
    shutil.copy2(p, artifact_metrics_dir / p.name)

print("Copied metrics into:", artifact_metrics_dir)

In [ ]:
# 16_push_full_merged_model_to_hub
# Chỉ push folder full merged đã pass local reload gate.
# delete_patterns=["*"] giúp xóa artifact cũ trên repo trước khi upload,
# tránh còn sót adapter_config/modules.json cũ từ pipeline Unsloth.

if PUSH_TO_HUB:
    api = HfApi()
    api.create_repo(repo_id=REPO_ID, repo_type="model", exist_ok=True)

    upload_kwargs = dict(
        folder_path=str(LOCAL_MERGED_DIR),
        repo_id=REPO_ID,
        repo_type="model",
        commit_message="Upload full merged SentenceTransformers BGE-M3 medical VI dense model",
    )

    if DELETE_REMOTE_FILES_BEFORE_UPLOAD:
        upload_folder_sig = inspect.signature(api.upload_folder)
        if "delete_patterns" not in upload_folder_sig.parameters:
            raise RuntimeError(
                "Installed huggingface_hub.HfApi.upload_folder does not support delete_patterns. "
                "Upgrade huggingface_hub or manually clear the target repo before uploading."
            )
        upload_kwargs["delete_patterns"] = ["*"]

    api.upload_folder(**upload_kwargs)
    print("Uploaded full merged model to Hub:", REPO_ID)
else:
    print("PUSH_TO_HUB=False. Skipped upload.")

In [ ]:
# 17_hub_reload_gate_and_final_eval
# Nếu muốn kiểm tra sạch nhất, restart runtime sau upload rồi chạy cell này.
# Trong cùng runtime, cell này vẫn dùng cache folder riêng để giảm nguy cơ dùng nhầm local folder.

if PUSH_TO_HUB:
    cleanup_cuda()

    if HUB_RELOAD_CACHE.exists():
        shutil.rmtree(HUB_RELOAD_CACHE)
    HUB_RELOAD_CACHE.mkdir(parents=True, exist_ok=True)

    try:
        hub_model = SentenceTransformer(
            REPO_ID,
            cache_folder=str(HUB_RELOAD_CACHE),
            model_kwargs={"torch_dtype": TORCH_DTYPE},
            trust_remote_code=False,
        )
    except TypeError:
        hub_model = SentenceTransformer(
            REPO_ID,
            cache_folder=str(HUB_RELOAD_CACHE),
        )

    hub_model.max_seq_length = MAX_SEQ_LENGTH

    hub_emb = smoke_test_model(hub_model, expected_dim=1024, name="hub_reloaded_model")

    hub_dev_metrics = dev_evaluator(hub_model)
    hub_test_metrics = test_evaluator(hub_model)

    print("HUB DEV METRICS")
    print(json.dumps(hub_dev_metrics, indent=2, ensure_ascii=False))
    print("HUB TEST METRICS")
    print(json.dumps(hub_test_metrics, indent=2, ensure_ascii=False))

    save_json(hub_dev_metrics, METRICS_DIR / "hub_dev_metrics.json")
    save_json(hub_test_metrics, METRICS_DIR / "hub_test_metrics.json")

    print("Hub reload gate passed.")
else:
    print("PUSH_TO_HUB=False. Skipped Hub reload gate.")

In [ ]:
# 18_final_artifact_audit_summary

summary = {
    "base_model": BASE_MODEL,
    "dataset_id": DATASET_ID,
    "repo_id": REPO_ID,
    "max_seq_length": MAX_SEQ_LENGTH,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "warmup_ratio": WARMUP_RATIO,
    "weight_decay": WEIGHT_DECAY,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "lora_target_modules": LORA_TARGET_MODULES,
    "train_rows": len(train_ds),
    "dev_rows": len(dev_ds),
    "test_rows": len(test_ds),
    "local_merged_dir": str(LOCAL_MERGED_DIR),
    "pushed_to_hub": PUSH_TO_HUB,
    "timestamp": datetime.utcnow().isoformat() + "Z",
    "versions": {
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "datasets": datasets.__version__,
        "peft": peft.__version__,
        "sentence_transformers": sentence_transformers.__version__,
        "huggingface_hub": huggingface_hub.__version__,
    },
}

summary_path = OUTPUT_ROOT / "run_summary.json"
save_json(summary, summary_path)

print("Run summary:")
print(json.dumps(summary, indent=2, ensure_ascii=False))

# Final hard audit on local artifact.
all_files = [p.relative_to(LOCAL_MERGED_DIR).as_posix() for p in LOCAL_MERGED_DIR.rglob("*") if p.is_file()]
assert any(f == "modules.json" for f in all_files)
assert any(f == "config_sentence_transformers.json" for f in all_files)
assert not any("adapter_model" in f or "adapter_config" in f for f in all_files)
assert any(re.search(r"(model|pytorch_model).*\.(safetensors|bin)$", f) for f in all_files)

print("FINAL AUDIT PASSED: local artifact is a full merged SentenceTransformers model, not adapter-only.")
print("Use:")
print(f'  model = SentenceTransformer("{REPO_ID}")')

## Checklist sau khi chạy notebook

Model chỉ được coi là hợp lệ nếu các cell sau pass:

- `13_save_full_merged_sentence_transformers_artifact`
  - Có `modules.json`.
  - Có `config_sentence_transformers.json`.
  - Có full model weights.
  - Không có `adapter_model.safetensors`.
  - Không có `adapter_config.json`.

- `14_local_reload_gate_and_eval`
  - `SentenceTransformer(LOCAL_MERGED_DIR)` load được.
  - Encode ra vector shape `(2, 1024)`.
  - Không có NaN/Inf.

- `17_hub_reload_gate_and_final_eval`
  - `SentenceTransformer(REPO_ID)` load được.
  - Encode ra vector shape `(2, 1024)`.
  - Không lỗi `sentence_transformers.base`.
  - Không cần Unsloth.
  - Không cần shim.